# **Kaggle competition -  Home Credit Risk Prediction - LightAutoML Train - Silver Medal**

This is my first Kaggle competition. While I have worked on datasets through the MIT Professional course, this is the first real world dataset. This dataset is complex and huge, hence I plan to take a systematic step by step approach. Understanding the dataset is of utmost importance for a successful data scientist. A good insight into data will help me make better decisions aboout the aggregation I would like to make and any feature engineering once I have a handle of all data and features I have used to achieve best possible results. Hence, I will be taking a slow and incremental change approach. This will not only make tracing changes easier, but also enhance my learning by letting me better understand what step leads to what change.

I have learned a lot from fellow Kagglers who are gracious in sharing their code as well as their knowledge. I have extensively refered to some of the following notebooks for my learning process.

Reference files for this is:

- https://www.kaggle.com/code/dksdms4/lb-0-565-improved-baseline-notebook
- https://www.kaggle.com/code/greysky/home-credit-baseline
- https://www.kaggle.com/code/ravi20076/homecredit-starter-inference-v1
- https://www.kaggle.com/code/peizhengwang/lb-0-57-mod-weight-pure-lgb
- https://www.kaggle.com/code/andreynesterov/home-credit-baseline-inference
- https://www.kaggle.com/code/andreynesterov/home-credit-baseline-training-lightautoml

### **Version information**

This version has lightautoml model using LGB, XGboost and CatBoost.

### **Install dependencies and libraries**

Link to lightautoml library: https://lightautoml.readthedocs.io/en/latest/pages/modules/generated/lightautoml.automl.presets.tabular_presets.TabularAutoML.html

Link to article on how to use dependencies in another kernel with internet off:
- https://towardsdatascience.com/easy-kaggle-offline-submission-with-chaining-kernels-30bba5ea5c4d

In [ ]:
%%time
# install the lightautoml library
!pip install -q --no-index --find-links=/kaggle/input/lightautoml-library-package/frozen_packages lightautoml

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing

# import further libraries
import matplotlib.pyplot as plt
import seaborn as sns

# library for cross validation
from sklearn.model_selection import cross_val_score, StratifiedKFold

# library for metrics
from sklearn.metrics import roc_auc_score

# library for AUTOML
from lightautoml.automl.presets.tabular_presets import TabularAutoML,TabularUtilizedAutoML
from lightautoml.dataset.roles import DatetimeRole
from lightautoml.tasks import Task

# library to read and write files
import pickle

# library to save and load models
import joblib

# display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# library for garbage collection
import gc  # since the data is huge here, regularly cleaning up will free up memory

# library to catch and ignore warnings
import warnings
warnings.filterwarnings("ignore")

# library for utility script containing various pipelines
import homecreditutility_v5 as hcu #version8

#### **Loading data onto dataframes**

In [ ]:
%%time
# load data
train_df = pd.read_csv("/kaggle/input/v8-data-prep-homecreditrisk2024/train_df.csv")

# setting categorical data as categories
with open('/kaggle/input/v8-data-prep-homecreditrisk2024/train_cat_cols.pkl', 'rb') as f:
     cat_cols = pickle.load(f)
        
train_df[cat_cols] = train_df[cat_cols].astype('category')
train_df.info()

### **Setting up lightautoml parameters**

In [ ]:
import torch

#Parameters and fix torch no of threads and numpy seeds
N_FOLDS = 5 # 4-fold cv
N_THREADS = 4  #threads
RANDOM_STATE=13 # fixed random state
TIMEOUT = 10*3600 #Time

np.random.seed(RANDOM_STATE)
torch.set_num_threads(N_THREADS)

In [ ]:
#initiate the task
task = Task('binary', metric='auc')

In [ ]:
#feature selection
roles = {
    'target':'target',
    'group': "WEEK_NUM",
    'drop':['case_id', 'WEEK_NUM'],
}

In [ ]:
#Automl
automl = TabularUtilizedAutoML(task = task,
                              timeout=TIMEOUT,
                              cpu_limit=N_THREADS,
                              gpu_ids = 'all',
                              reader_params = {'n_jobs':N_THREADS, 'cv': N_FOLDS},
                              general_params = {'use_algos':[['lgb', 'lgb_tuned','cb', 'cb_tuned']]},
                              tuning_params = {'max_tuning_time':60*60},)
                             

In [ ]:
#prediction
pred = automl.fit_predict(train_df,roles=roles, verbose=2)
print('pred:\n{}\nShape = {}'.format(pred[:10],pred.shape))

In [ ]:
# save automl model
joblib.dump(automl, 'automl.joblib')

### **Function to determine gini_stability**

In [ ]:
def gini_stability(base, w_fallingrate=88.0, w_resstd=-0.5):
    gini_in_time = base.loc[:, ["WEEK_NUM", "target", "predict"]]\
        .sort_values("WEEK_NUM")\
        .groupby("WEEK_NUM")[["target", "predict"]]\
        .apply(lambda x: 2*roc_auc_score(x["target"], x["predict"])-1).tolist()

    x = np.arange(len(gini_in_time))
    y = gini_in_time
    a, b = np.polyfit(x, y, 1)
    y_hat = a*x + b
    residuals = y - y_hat
    res_std = np.std(residuals)
    avg_gini = np.mean(gini_in_time)
    return avg_gini + w_fallingrate * min(0, a) + w_resstd * res_std

### **Function to determine batch predictions for bigger test dataset**

In [ ]:
def predict_proba_in_batches(model, data, batch_size=100000):
    num_samples = len(data)
    num_batches = int(np.ceil(num_samples / batch_size))
    probabilities = np.zeros((num_samples,))

    for batch_idx in range(num_batches):
        print(f"Processing batch: {batch_idx+1}/{num_batches}")
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, num_samples)
        X_batch = data.iloc[start_idx:end_idx].reset_index(drop=True)
        batch_probs = model.predict(X_batch).data[:, 0]
        probabilities[start_idx:end_idx] = batch_probs

    return probabilities

## **train_df prediction using batch prediction and gini score**

In [ ]:
%%time
# predict using the trained gbm model
train_cols = train_df.columns.to_list()
train_cols.remove('case_id')
train_cols.remove('target')
train_cols.remove('WEEK_NUM')
y_pred = pd.Series(predict_proba_in_batches(automl, train_df[train_cols]), index=train_df.index)

# display AUC score for the train and validation datasets
print(f'The AUC score on the train set is: {roc_auc_score(train_df["target"], y_pred)}')

In [ ]:
%%time
pred_df = train_df[["WEEK_NUM", "target"]].copy()
pred_df["predict"] = y_pred

# finding the stability scores for train and validation datasets
stability_score_train = gini_stability(pred_df)

# display the stability scores for train and validation dataset
print(f'The stability score on the train set is: {stability_score_train}')